In [242]:
import pandas as pd
import re
from datetime import datetime, date

In [243]:
votes_backup = pd.read_csv("baarbierians_voting_backup.csv")

In [244]:
votes_backup.head()

,filled_by,date,category,winner,in_pub,points,winner_num
0,Andrew,2025-06-06,Captains Performance,Andreas,True,1,1
1,Andrew,2025-06-06,Drama Queen,Bats,False,-2,1
2,Andrew,2025-06-06,Duffer,Hubert,True,-1,1
3,Andrew,2025-06-06,Duffer,Panu,False,-2,2
4,Andrew,2025-06-06,Goal of the Night,Andreas,True,1,1


In [289]:
votes_backup["date"] = pd.to_datetime(votes_backup["date"]).dt.date

In [290]:
last_backup_date = votes_backup["date"].max()
last_backup_date

datetime.date(2025, 6, 6)

In [246]:
chat = open("_chat.txt", "r").read()

In [247]:
messages = chat.split('\n[')

In [248]:
chat_voting = [x for x in messages if "Voting results" in x]

In [250]:
chat_voting = [re.search('(?<=\] ).*', re.sub("\n", ";", x))[0].replace(u"~\u202f", "").replace(u"\u200e<This message was edited>", "").strip() for x in chat_voting]

In [251]:
chat_voting

['Andrew Irvine: Voting results 28.7:;Save of the Night: Didi(1);Goal of the night: Didi (1);Skill moment: Finn(1);Worst tackle: Guy (-1);Duffer: Anar (-2);Golden Goal: Jura (0);Captain‘s Performance: Andy L. (1);Drama Queen: Martial (-2);Greedy Bastard: Peter P (-2);\u200e[28.07.2023, 23:19:08] Gerd Wichmann: Amazing start of the season👍 Nur der HSV \u200eimage omitted;\u200e[29.07.2023, 13:30:13] Anar Habib: \u200eimage omitted',
 'Andrew Irvine: Voting results 4.8:;Save of the Night: Hubert (1);Goal of the night: Hogie 007 (1);Skill moment: Christoph (1);Worst tackle: Ian P (-2);Duffer: Andrew(-1);Golden Goal: Jura (0),  Oscar (0);Captain‘s Performance: Mark B (1);Drama Queen: Jura (-2);Greedy Bastard: Peter P (-2)',
 'Andrew Irvine: Voting results 11.8:;Save of the Night:Rodrigo (1);Goal of the night: Maris(1);Skill moment: Jura (1);Worst tackle: Simon (-2);Duffer: Andrew (-1);Golden Goal: Martial (0);Captain‘s Performance: Scott (1);Drama Queen: Simon(-2);Greedy Bastard: Andy (-2)

In [252]:
chat_voting = [x for x in chat_voting if re.search('(?<=Voting results )\d{2}.\d{2}.\d{4}', x) and not x.startswith("Chris")]

In [253]:
CATEGORIES = {"Goal of the Night": True,
            "Save of the Night": True,
            "Skill Moment": True,
            "Worst Tackle": False,
            "Duffer": False,
            "Drama Queen": False,
            "Greedy Bastard": False,
            "Golden Goal": True,
            "Captain's Performance": True}

In [254]:
chat_votes = [
    {
        "filled_by" : re.search('[A-Za-z]+(?=[- :])', x)[0],
        "date" : datetime.strptime(re.search('(?<=Voting results )\d{2}.\d{2}.\d{4}', x)[0], "%d.%m.%Y").date(),
        "category": category,
        "winner": (
            (re.search(f"(?<={category}:)[^;]+", x)[0])
            .strip()
            .replace("&", ",")
            .replace("The Americans (John B (1), Chris Davis (1))", "John B (1), Chris Davis (1)")
            .replace("The whole Wichmann family (-1/-2)", "Finn W (-1), Gerd W (-1), Gerd's wife (-2)")
            .split(","))
    }
    for x in chat_voting
    for category in CATEGORIES.keys()
    if datetime.strptime(re.search('(?<=Voting results )\d{2}.\d{2}.\d{4}', x)[0], "%d.%m.%Y").date() > last_backup_date
]

In [255]:
chat_votes[:2]

[{'filled_by': 'Andrew',
  'date': datetime.date(2025, 6, 20),
  'category': 'Goal of the Night',
  'winner': ['Hubert (1)']},
 {'filled_by': 'Andrew',
  'date': datetime.date(2025, 6, 20),
  'category': 'Save of the Night',
  'winner': ['Maris (0)']}]

In [310]:
chat_votes_df = pd.DataFrame(chat_votes).explode("winner")

In [311]:
chat_votes_df

,filled_by,date,category,winner
0,Andrew,2025-06-20,Goal of the Night,Hubert (1)
1,Andrew,2025-06-20,Save of the Night,Maris (0)
2,Andrew,2025-06-20,Skill Moment,Alexandru (0)
3,Andrew,2025-06-20,Worst Tackle,Finn W (-2)
4,Andrew,2025-06-20,Duffer,Alistair (-1)
...,...,...,...,...
409,Andrew,2026-07-24,Duffer,Not Mike (-2)
410,Andrew,2026-07-24,Drama Queen,Alistair(-1)
411,Andrew,2026-07-24,Greedy Bastard,Stevie(-2)
412,Andrew,2026-07-24,Golden Goal,Tilo (0)


In [378]:
chat_votes_df["filled_by"] = chat_votes_df["filled_by"].replace("Giovanni", "Gio").replace("Jack", "Jack P")

In [313]:
chat_votes_df.loc[~chat_votes_df["winner"].str.contains(".+\(.+\)"), "winner"] = chat_votes_df.loc[~chat_votes_df["winner"].str.contains(".+\(.+\)"), "winner"] + " (-1)"

In [314]:
chat_votes_df['winner'] = chat_votes_df["winner"].str.replace("( - draw in voting)|(\(new Chris nickname\))|(\(definitely not Mike F\))", "", regex=True)

In [315]:
chat_votes_df["points"] = chat_votes_df["winner"].map(lambda x: int(re.search("(?<=\().+(?=\))", x)[0]))

In [316]:
chat_votes_df["points"].drop_duplicates()

0    1
1    0
3   -2
4   -1
Name: points, dtype: int64

In [317]:
chat_votes_df["in_pub"] = chat_votes_df["points"].abs().eq(1)

In [318]:
chat_votes_df["winner"] = chat_votes_df["winner"].str.replace("\([-\+]?\d\)","", regex=True).str.strip()

In [319]:
chat_votes_df["winner_num"] = chat_votes_df.index.to_series().groupby(level=0).cumcount() + 1

In [345]:
tour_votes1 = pd.DataFrame([
    {
        "category": "Goal of the Night",
        "winner": "Gio",
        "points": 1
    },
    {
        "category": "Save of the Night",
        "winner": "Hubert",
        "points": 1
    },
    {
        "category": "Skill Moment",
        "winner": "Mama Gio",
        "points": 0
    },
    {
        "category": "Worst tackle",
        "winner": "Simon",
        "points": -1
    },
    {
        "category": "Duffer",
        "winner": "Mark B",
        "points": -1
    },
    {
        "category": "Drama Queen",
        "winner": "Gerd W",
        "points": -1
    },
    {
        "category": "Greedy Bastard",
        "winner": "Gio",
        "points": -1
    },
    {
        "category": "Golden Goal",
        "winner": "Lloyd",
        "points": 1
    },
    {
        "category": "Captain's Performance",
        "winner": "Scott",
        "points": 1
    },
])
tour_votes1["date"] = date.fromisoformat("2025-09-12")

In [346]:
tour_votes2 = pd.DataFrame([
    {
        "category": "Goal of the Night",
        "winner": "Francesco",
        "points": 1
    },
    {
        "category": "Save of the Night",
        "winner": "Mark B",
        "points": 1
    },
    {
        "category": "Skill Moment",
        "winner": "Salvio",
        "points": 1
    },
    {
        "category": "Worst tackle",
        "winner": "Gokhan",
        "points": -1
    },
    {
        "category": "Duffer",
        "winner": "Andrew",
        "points": -1
    },
    {
        "category": "Drama Queen",
        "winner": "Lloyd",
        "points": -1
    },
    {
        "category": "Greedy Bastard",
        "winner": "Gio",
        "points": -1
    },
    {
        "category": "Golden Goal",
        "winner": "Jura",
        "points": 1
    },
    {
        "category": "Captain's Performance",
        "winner": "Jura",
        "points": 1
    },
])
tour_votes2["date"] = date.fromisoformat("2025-09-13")

In [347]:
tour_votes = pd.concat([tour_votes1, tour_votes2])
tour_votes["filled_by"] = "Andrew"
tour_votes["winner_num"] = 1
tour_votes["in_pub"] = tour_votes["points"].abs().eq(1)

In [348]:
tour_votes

,category,winner,points,date,filled_by,winner_num,in_pub
0,Goal of the Night,Gio,1,2025-09-12,Andrew,1,True
1,Save of the Night,Hubert,1,2025-09-12,Andrew,1,True
2,Skill Moment,Mama Gio,0,2025-09-12,Andrew,1,False
3,Worst tackle,Simon,-1,2025-09-12,Andrew,1,True
4,Duffer,Mark B,-1,2025-09-12,Andrew,1,True
5,Drama Queen,Gerd W,-1,2025-09-12,Andrew,1,True
6,Greedy Bastard,Gio,-1,2025-09-12,Andrew,1,True
7,Golden Goal,Lloyd,1,2025-09-12,Andrew,1,True
8,Captain's Performance,Scott,1,2025-09-12,Andrew,1,True
0,Goal of the Night,Francesco,1,2025-09-13,Andrew,1,True


In [379]:
votes = pd.concat([votes_backup, chat_votes_df, tour_votes])

In [390]:
votes[['points', "winner_num"]] = votes[['points', "winner_num"]].map(int)
votes['in_pub'] = votes['in_pub'].map(str)

In [381]:
votes["winner"].drop_duplicates().sort_values().to_list()

['Abdel',
 'Abduhla',
 'Adam - other',
 'Adam M',
 'Aidan',
 'Akos',
 'Akos’s kid',
 'Akos’s wife',
 'Alejandro',
 'Alexandru',
 'Alfredo',
 'Ali',
 'Ali jee',
 'Alistair',
 'American Chris',
 'Anar',
 'Andis',
 'Andre',
 'Andre V  you left too early',
 'Andreas',
 'Andrew',
 'Andrew - other',
 'Andrew Beatie',
 'Andrew Dickmann',
 'Andrew S',
 'André C',
 'André Cecílio',
 'André V',
 'Andy',
 'Andy (younger)',
 'Andy K',
 "Andy's boyfriend",
 "Andy's mum",
 'Anton',
 'Audi Guy',
 'Bats',
 'Bats’ Uncle',
 'Bert',
 'Brauerei',
 'Captain Tom',
 'Carlo',
 'Carlo’s six pack',
 'Chris Davis',
 'Chris S',
 'Chris Shires',
 'Christian',
 'Christoph',
 'Colin B',
 'Connor',
 'Connor H',
 'Constantijn',
 'Cordon Bleu',
 'Corona virus',
 'Crossbar',
 'Cédric',
 'Damian',
 'Dan W',
 'Dan Y',
 'Daniel',
 'Dave McKimm',
 'Davis',
 'Didi',
 'Didier C',
 'Dino',
 'Dow Maris',
 'Edi',
 'Emilio',
 'Emily Ramsey',
 'Emma Mason',
 'Fat Leigh',
 'Ferhat',
 'Finlay H',
 'Finn',
 'Finn H',
 'Finn W',
 'Fra

In [382]:
votes["winner"] = votes["winner"].replace("Carlo's six pack", "Carlo").replace("Chris Davis", "American Chris").replace("Chris Shires", "Chris S").replace("Davis", "American Chris").replace("Fat Leigh", "Leigh Q").replace("Finn", "Finn W").replace("Gerd W", "Gerd").replace("Gary on himself", "Gary").replace("Viktor", "Victor").replace("Vitor", "Victor").replace("Not Mike (Michal)", "Not Mike").replace("Ali jee", "Ali").replace("Andre V  you left too early", "André V").replace("Andre", "André V").replace("Connor", "Connor H").replace("Hubert  - for coming to the pub after football", "Hubert").replace("JCP", "Julio").replace("Julie P", "Julie Popple").replace("Leigh (Skinny)", "Lee (skinny)").replace("Lee", "Lee (skinny)").replace("Lloyd 🤕", "Lloyd").replace("Niall‘s head", "Niall").replace("Nick", "Nik").replace("Scott‘s balls", "Scott").replace("Jack", "Jack P")

In [383]:
votes["winner"].drop_duplicates().sort_values().to_list()

['Abdel',
 'Abduhla',
 'Adam - other',
 'Adam M',
 'Aidan',
 'Akos',
 'Akos’s kid',
 'Akos’s wife',
 'Alejandro',
 'Alexandru',
 'Alfredo',
 'Ali',
 'Alistair',
 'American Chris',
 'Anar',
 'Andis',
 'Andreas',
 'Andrew',
 'Andrew - other',
 'Andrew Beatie',
 'Andrew Dickmann',
 'Andrew S',
 'André C',
 'André Cecílio',
 'André V',
 'Andy',
 'Andy (younger)',
 'Andy K',
 "Andy's boyfriend",
 "Andy's mum",
 'Anton',
 'Audi Guy',
 'Bats',
 'Bats’ Uncle',
 'Bert',
 'Brauerei',
 'Captain Tom',
 'Carlo',
 'Carlo’s six pack',
 'Chris S',
 'Christian',
 'Christoph',
 'Colin B',
 'Connor H',
 'Constantijn',
 'Cordon Bleu',
 'Corona virus',
 'Crossbar',
 'Cédric',
 'Damian',
 'Dan W',
 'Dan Y',
 'Daniel',
 'Dave McKimm',
 'Didi',
 'Didier C',
 'Dino',
 'Dow Maris',
 'Edi',
 'Emilio',
 'Emily Ramsey',
 'Emma Mason',
 'Ferhat',
 'Finlay H',
 'Finn H',
 'Finn W',
 'Francesco',
 'Francisco',
 'Frank H',
 'Gary',
 'Gary - other',
 'George',
 'Gerd',
 "Gerd's wife",
 'Gio',
 'Gireg',
 'Gokhan',
 'Guy

In [384]:
votes = votes.sort_values(["date", "category", "winner"])

In [385]:
votes["date"].drop_duplicates().sort_values()

3866    2014-01-03
3856    2014-01-10
3849    2014-01-17
3842    2014-01-24
3837    2014-01-31
           ...    
377     2026-06-26
386     2026-07-03
395     2026-07-10
404     2026-07-17
413     2026-07-24
Name: date, Length: 510, dtype: object

In [387]:
votes["date"].drop_duplicates().sort_values().to_list()

[datetime.date(2014, 1, 3),
 datetime.date(2014, 1, 10),
 datetime.date(2014, 1, 17),
 datetime.date(2014, 1, 24),
 datetime.date(2014, 1, 31),
 datetime.date(2014, 2, 7),
 datetime.date(2014, 2, 14),
 datetime.date(2014, 2, 21),
 datetime.date(2014, 2, 28),
 datetime.date(2014, 3, 7),
 datetime.date(2014, 3, 14),
 datetime.date(2014, 3, 21),
 datetime.date(2014, 3, 28),
 datetime.date(2014, 4, 4),
 datetime.date(2014, 4, 11),
 datetime.date(2014, 4, 25),
 datetime.date(2014, 5, 2),
 datetime.date(2014, 5, 9),
 datetime.date(2014, 5, 16),
 datetime.date(2014, 5, 23),
 datetime.date(2014, 5, 30),
 datetime.date(2014, 6, 6),
 datetime.date(2014, 6, 13),
 datetime.date(2014, 6, 20),
 datetime.date(2014, 7, 11),
 datetime.date(2014, 7, 18),
 datetime.date(2014, 7, 25),
 datetime.date(2014, 8, 8),
 datetime.date(2014, 8, 15),
 datetime.date(2014, 8, 22),
 datetime.date(2014, 8, 29),
 datetime.date(2014, 9, 5),
 datetime.date(2014, 9, 12),
 datetime.date(2014, 9, 19),
 datetime.date(2014, 10

In [391]:
votes

,filled_by,date,category,winner,in_pub,points,winner_num
3866,Scott,2014-01-03,Goal of the Night,Paul L,True,1,1
3867,Scott,2014-01-03,Save of the Night,John B,False,0,1
3856,Scott,2014-01-10,Captains Performance,John B,True,1,1
3858,Scott,2014-01-10,Duffer,Didier C,True,-1,1
3859,Scott,2014-01-10,Duffer,Mark L,True,-1,2
...,...,...,...,...,...,...,...
412,Andrew,2026-07-24,Golden Goal,Tilo,False,0,1
411,Andrew,2026-07-24,Greedy Bastard,Stevie,False,-2,1
406,Andrew,2026-07-24,Save of the Night,Hubert,True,1,1
407,Andrew,2026-07-24,Skill Moment,Neil H,False,0,1


In [388]:
votes["category"] = votes["category"].replace("Captain's Performance", "Captains Performance")

In [392]:
votes.to_csv(f"baarbierians_voting_backup_{date.today().isoformat()}.csv", index=False)